# Matplotlib — Zero to Hero Worksheet
Read `concept_notes.md` and `diagrams.md` first. Every plot in the ChromaDB lab was made with exactly the patterns you'll build here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
print("Ready")

## 1. Your first plot — line chart

In [ ]:
x = np.linspace(0, 2 * np.pi, 100)
y_sin = np.sin(x)
y_cos = np.cos(x)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, y_sin, label="sin(x)", color="blue")
ax.plot(x, y_cos, label="cos(x)", color="red", linestyle="--")
ax.set_title("Sine and Cosine")
ax.set_xlabel("x (radians)")
ax.set_ylabel("y")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

### Observe:
- Change `linestyle="--"` to `":"` or `"-."`  — what changes?
- Change `color="red"` to `color="#FF8C00"` — matplotlib accepts hex colors.
- Add `ax.set_xlim(0, np.pi)` — what happens to the plot range?

## 2. Scatter plot — the one used most in the ChromaDB lab

In [ ]:
np.random.seed(42)

# Simulate 3 topic clusters in 2D (like a PCA result)
n = 30
topics = ["cooking"] * n + ["finance"] * n + ["sports"] * n
x = np.concatenate([np.random.randn(n) + 0,     # cooking cluster at x~0
                     np.random.randn(n) + 4,     # finance cluster at x~4
                     np.random.randn(n) + 2])    # sports cluster at x~2
y = np.concatenate([np.random.randn(n) + 0,
                     np.random.randn(n) + 3,
                     np.random.randn(n) - 2])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: all one color (hard to see structure) ---
axes[0].scatter(x, y, alpha=0.6, s=40)
axes[0].set_title("All one color — structure invisible")

# --- Right: colored by topic (structure visible) ---
colors = {"cooking": "red", "finance": "blue", "sports": "green"}
for topic in set(topics):
    idx = [i for i, t in enumerate(topics) if t == topic]
    axes[1].scatter(x[idx], y[idx], label=topic,
                     color=colors[topic], alpha=0.7, s=50)
axes[1].legend()
axes[1].set_title("Colored by topic — clusters visible")

plt.tight_layout()
plt.show()

### Observe:
- The left plot and right plot use the same x/y data. The only difference is coloring. This is the core insight of the ChromaDB lab's visualization — the structure is always there, you just need to encode the labels as color to see it.
- Change `alpha=0.7` to `alpha=0.2` and `alpha=1.0` — when would you use each?

## 3. Bar chart and histogram — comparing distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart — categorical comparison
models = ["Qwen3-14B", "Qwen3-30B", "Mistral", "Llama-70B"]
latency_ms = [120, 310, 180, 520]
colors_bar = ["#2196F3", "#FF9800", "#4CAF50", "#9C27B0"]

axes[0].bar(models, latency_ms, color=colors_bar, edgecolor="white", width=0.6)
axes[0].set_title("Model latency comparison")
axes[0].set_ylabel("Latency (ms)")
axes[0].set_xlabel("Model")
for i, v in enumerate(latency_ms):
    axes[0].text(i, v + 5, str(v), ha="center", fontsize=10)  # add value labels

# Histogram — distribution of cosine similarity scores
np.random.seed(42)
similarity_scores = np.concatenate([
    np.random.normal(0.85, 0.05, 50),   # high-similarity pairs (same topic)
    np.random.normal(0.40, 0.10, 200),  # low-similarity pairs (different topic)
])
axes[1].hist(similarity_scores, bins=30, color="#2196F3", edgecolor="white", alpha=0.8)
axes[1].axvline(x=0.7, color="red", linestyle="--", label="threshold = 0.7")
axes[1].set_title("Distribution of cosine similarity scores")
axes[1].set_xlabel("Cosine similarity")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.show()

### Observe:
- The histogram shows two bumps — a bimodal distribution. Why would cosine similarities between random document pairs show this shape?
- The `axvline` red dashed line at 0.7 simulates a retrieval threshold — pairs above this line would be considered 'similar enough to retrieve'. How many pairs in the histogram fall above this threshold?

## 4. Subplots grid — exactly what the ChromaDB lab uses

In [ ]:
np.random.seed(42)
n_points = 55

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Simulate PCA coordinates for 55 docs, 8 topics
topic_centers = [(0, 0), (3, 0), (6, 0), (1, 3), (4, 3), (7, 3), (2, 6), (5, 6)]
topic_names = ["cooking", "sports", "finance", "travel",
               "music", "history", "medicine", "programming"]
all_x, all_y, all_labels, all_types = [], [], [], []

for i, (cx, cy) in enumerate(topic_centers):
    n = 6
    all_x.extend(np.random.randn(n) * 0.5 + cx)
    all_y.extend(np.random.randn(n) * 0.5 + cy)
    all_labels.extend([topic_names[i]] * n)
    all_types.extend(["core"] * n)

all_x.extend([topic_centers[0][0] + 0.1, topic_centers[2][0] + 0.1])
all_y.extend([topic_centers[0][1] + 0.1, topic_centers[2][1] + 0.1])
all_labels.extend(["cooking", "finance"])
all_types.extend(["near_duplicate", "near_duplicate"])

all_x = np.array(all_x); all_y = np.array(all_y)

# Top-left: all gray (baseline)
axes[0, 0].scatter(all_x, all_y, color="gray", alpha=0.6, s=30)
axes[0, 0].set_title("All dots, no color info")

# Top-right: colored by topic
cmap = plt.get_cmap("tab10")
for i, topic in enumerate(topic_names):
    idx = [j for j, l in enumerate(all_labels) if l == topic]
    axes[0, 1].scatter(all_x[idx], all_y[idx], color=cmap(i), label=topic, s=40, alpha=0.8)
axes[0, 1].legend(fontsize=7, loc="upper left")
axes[0, 1].set_title("Colored by topic")

# Bottom-left: colored by type
type_colors = {"core": "lightgray", "near_duplicate": "red"}
for t, c in type_colors.items():
    idx = [j for j, tp in enumerate(all_types) if tp == t]
    axes[1, 0].scatter(all_x[idx], all_y[idx], color=c, label=t,
                        s=80 if t != "core" else 30, alpha=0.9)
axes[1, 0].legend()
axes[1, 0].set_title("Colored by type (near-duplicates in red)")

# Bottom-right: annotated specific points
axes[1, 1].scatter(all_x, all_y, color="steelblue", alpha=0.4, s=30)
special = [i for i, t in enumerate(all_types) if t == "near_duplicate"]
for idx in special:
    axes[1, 1].scatter(all_x[idx], all_y[idx], color="red", s=100, zorder=5)
    axes[1, 1].annotate(f"dup_{idx}", (all_x[idx], all_y[idx]),
                         xytext=(5, 5), textcoords="offset points", fontsize=8)
axes[1, 1].set_title("Annotated near-duplicates")

plt.suptitle("Four views of the same 55-document embedding space", y=1.02)
plt.tight_layout()
plt.show()

### Observe:
- All 4 plots use the exact same x/y coordinates. What's the *only* thing that changes between them?
- Top-left vs top-right: does the structure become clearer, or is it just prettier? What does this mean about what color encoding is doing?
- Bottom-right: try changing `xytext=(5, 5)` to `xytext=(20, 20)` — what happens to the annotation position?

## 5. Heatmap — confusion matrices and similarity matrices

In [ ]:
np.random.seed(42)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: confusion matrix (classification results)
conf_matrix = np.array([
    [45,  3,  2],
    [ 4, 38,  8],
    [ 1,  5, 44],
])
im0 = axes[0].imshow(conf_matrix, cmap="Blues")
axes[0].set_title("Confusion Matrix")
axes[0].set_xlabel("Predicted label")
axes[0].set_ylabel("True label")
axes[0].set_xticks([0, 1, 2])
axes[0].set_yticks([0, 1, 2])
axes[0].set_xticklabels(["cooking", "finance", "sports"])
axes[0].set_yticklabels(["cooking", "finance", "sports"])
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, conf_matrix[i, j], ha="center", va="center",
                      color="black" if conf_matrix[i, j] < 30 else "white")
plt.colorbar(im0, ax=axes[0])

# Right: cosine similarity matrix (5 sample embedding vectors)
vecs = np.random.randn(5, 8)
vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)   # normalize
sim_matrix = vecs @ vecs.T   # cosine similarity = dot product of normalized vectors
im1 = axes[1].imshow(sim_matrix, cmap="RdYlGn", vmin=-1, vmax=1)
axes[1].set_title("Cosine Similarity Matrix
(5 embedding vectors)")
axes[1].set_xticks(range(5))
axes[1].set_yticks(range(5))
axes[1].set_xticklabels([f"doc_{i}" for i in range(5)])
axes[1].set_yticklabels([f"doc_{i}" for i in range(5)])
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

### Observe:
- In the confusion matrix, the diagonal shows correct predictions. Which class has the most mistakes?
- In the similarity matrix, the diagonal is always exactly 1.0. Why?
- The similarity matrix is symmetric (mirrored across the diagonal). Why is `sim(doc_0, doc_3)` always equal to `sim(doc_3, doc_0)`?

## 6. The teaser: making a dense scatter readable with annotations

In [ ]:
np.random.seed(42)

# Simulate the near-duplicate pair problem from concept_notes.md
n = 40
x = np.random.randn(n) * 2    # all same cluster
y = np.random.randn(n) * 2

near_dup_x = x[0] + np.random.randn(2) * 0.05   # near-duplicates very close to doc[0]
near_dup_y = y[0] + np.random.randn(2) * 0.05

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Problem: dense blob, near-duplicates invisible
axes[0].scatter(np.append(x, near_dup_x), np.append(y, near_dup_y),
                 color="steelblue", s=30, alpha=0.6)
axes[0].set_title("Problem: near-duplicates invisible")

# Fix 1: color by type
axes[1].scatter(x, y, color="steelblue", s=30, alpha=0.6, label="core")
axes[1].scatter(near_dup_x, near_dup_y, color="red", s=120,
                 alpha=1.0, zorder=5, marker="*", label="near_duplicate")
axes[1].legend()
axes[1].set_title("Fix 1: color encoding by type")

# Fix 2: color + annotation + alpha for overlap
axes[2].scatter(x, y, color="steelblue", s=30, alpha=0.3, label="core")
axes[2].scatter([x[0]], [y[0]], color="orange", s=150, zorder=5,
                 marker="D", label="original (doc_0)")
axes[2].scatter(near_dup_x, near_dup_y, color="red", s=120,
                 alpha=1.0, zorder=5, marker="*", label="near_duplicate")
for i, (nx, ny) in enumerate(zip(near_dup_x, near_dup_y)):
    axes[2].annotate(f"dup_{i+1}", (nx, ny),
                      xytext=(8, 8), textcoords="offset points", fontsize=9,
                      arrowprops=dict(arrowstyle="->", color="red"))
axes[2].legend(fontsize=8)
axes[2].set_title("Fix 2: color + alpha + annotation")

plt.suptitle("Three views of the same data — technique changes what you see", y=1.02)
plt.tight_layout()
plt.show()

### Final exercise
Load the practice dataset embeddings from the ChromaDB lab into Python (`pd.read_excel('chroma_practice_lab/dataset/practice_dataset.xlsx')`) and make your own version of this 4-subplot grid using:
- PCA coordinates from `sklearn.decomposition.PCA(n_components=2).fit_transform(vectors)`
- Color by `topic` column in one subplot, by `type` column in another
- Annotate the bridge and outlier documents by id

**This is the exact plot that lives in the ChromaDB lab — now you built it yourself from scratch.**